In [181]:
# loading env variables
from dotenv import load_dotenv

# splitting into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# prompt template
from langchain_core.prompts import ChatPromptTemplate

# structure output prompt
from pydantic import BaseModel, Field ,field_validator # Field enfroce advance constraints,guides ai using desc
from typing import List,Optional

# parsing
from langchain_core.output_parsers import PydanticOutputParser

# feature extractor

# llm 
from langchain_ollama import ChatOllama

load_dotenv()

True

In [182]:
dir(Field)

['__annotations__',
 '__builtins__',
 '__call__',
 '__class__',
 '__closure__',
 '__code__',
 '__defaults__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__globals__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__kwdefaults__',
 '__le__',
 '__lt__',
 '__module__',
 '__name__',
 '__ne__',
 '__new__',
 '__qualname__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__type_params__']

In [183]:
class UserReportSchema(BaseModel):
    # 1. Incident Details (What happened)
    offence_description: str = Field(
        description="A plain-text description of what happened (e.g., 'Bank Fraud', 'Stolen Wallet', 'Online Scam')"
    )
    
    # 2. Timeline (When it happened)
    approximate_date: Optional[str] = Field(
        default=None,
        description="The date or date-range when the incident took place (e.g., 'Tuesday', 'Between 2014 and 2016')"
    )
    approximate_time: Optional[str] = Field(
        default=None,
        description="The time of day the incident occurred if known"
    )

    # 3. Location (Where it happened)
    location_name: Optional[str] = Field(
        default=None,
        description="The name of the house, business, shop, or street where it occurred (e.g., 'Lenin Sarani')"
    )
    city: Optional[str] = Field(
        default=None,
        description="The city or town name"
    )
    state: Optional[str] = Field(
        default=None,
        description="The state name"
    )

    # 4. User Information (Who is reporting)
    reporter_full_name: Optional[str] = Field(
        default=None,
        description="The full name of the person reporting the incident"
    )
    reporter_occupation: Optional[str] = Field(
        default=None,
        description="The reporter's job title or profession"
    )

    @field_validator('*',mode='before')
    @classmethod
    def clean_placeholders(cls,value):
        INVALID_VALUES = {
            "Not provided",
            "Not mentioned",
            "Unknown",
            "N/A",
            "ReporterName",
            "CityName",
            "StateName",
            "Occupation"
        }

        if isinstance(value,str) and value.strip() in INVALID_VALUES:
            return None
        
        return value


In [184]:
parser = PydanticOutputParser(pydantic_object=UserReportSchema)

In [185]:
prompt = ChatPromptTemplate.from_messages([
    (
        'system','''
Extract important information from the paragraph.
Return ONLY a raw JSON object matching the requested schema. 

If information is not present,
return null.

Never invent placeholder values.
Never output CityName, StateName, ReporterName, etc.
{format_instructions}
'''
    ),
    (
        'human',"{incident}"
    )
])

In [186]:
incident = 'My bike got stolen near the mall'

In [187]:
incident

'My bike got stolen near the mall'

In [188]:
final_prompt =  prompt.invoke(
    {
        'incident':incident,
        'format_instructions':parser.get_format_instructions()
    }
)

In [189]:
llm = ChatOllama(
    model='mistral:7b',
    temperature=0
)

In [190]:
response = llm.invoke(final_prompt)

In [191]:
response.content

' ```\n{\n  "offence_description": "Stolen Bike",\n  "location_name": "near the mall",\n  "city": null,\n  "state": null,\n  "reporter_full_name": null,\n  "reporter_occupation": null,\n  "approximate_date": null,\n  "approximate_time": null\n}\n```'

In [192]:
final_data = parser.parse(response.content)

In [193]:
final_data

UserReportSchema(offence_description='Stolen Bike', approximate_date=None, approximate_time=None, location_name='near the mall', city=None, state=None, reporter_full_name=None, reporter_occupation=None)

In [194]:
data_dict = final_data.model_dump()

In [195]:
data_dict

{'offence_description': 'Stolen Bike',
 'approximate_date': None,
 'approximate_time': None,
 'location_name': 'near the mall',
 'city': None,
 'state': None,
 'reporter_full_name': None,
 'reporter_occupation': None}

In [196]:
missing = [
    field
    for field,value in data_dict.items()
    if value is None
]

In [197]:
missing

['approximate_date',
 'approximate_time',
 'city',
 'state',
 'reporter_full_name',
 'reporter_occupation']

In [ ]:
QUESTION_MAP = {
    "approximate_date": "On what date did the incident occur?",
    "approximate_time": "At approximately what time did the incident occur?",
    "location_name" : "At what location did the incident occur?",
    "city": "In which city did the incident occur?",
    "state": "In which state did the incident occur?",
    "reporter_full_name": "Please provide your full name.",
    "reporter_occupation": "What is your occupation?"
}